In [1]:
%pip install --upgrade openai openai-agents python-dotenv

Note: you may need to restart the kernel to use updated packages.


c:\Users\estev\OneDrive\Área de Trabalho\IT3025C Labs\Intro-to-Agentic-AI-Final-Project-main\Intro-to-Agentic-AI-Final-Project-main\.venv\Scripts\python.exe: No module named pip


In [4]:
import sys

print(sys.executable)

c:\Users\estev\OneDrive\Área de Trabalho\IT3025C Labs\Intro-to-Agentic-AI-Final-Project-main\Intro-to-Agentic-AI-Final-Project-main\.venv\Scripts\python.exe


In [2]:
import sys
import subprocess

subprocess.check_call([
    "uv", "pip", "install",
    "--python", sys.executable,
    "--link-mode", "copy",
    "openai",
    "openai-agents",
    "python-dotenv",
    "gradio",
])

print("Packages installed successfully.")

CalledProcessError: Command '['uv', 'pip', 'install', '--python', 'c:\\Users\\estev\\OneDrive\\Área de Trabalho\\IT3025C Labs\\Intro-to-Agentic-AI-Final-Project-main\\Intro-to-Agentic-AI-Final-Project-main\\.venv\\Scripts\\python.exe', '--link-mode', 'copy', 'openai', 'openai-agents', 'python-dotenv', 'gradio']' returned non-zero exit status 2.

In [ ]:
import os
import json

from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncAzureOpenAI

from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    SQLiteSession,
    set_tracing_disabled,
)

print("Imports loaded successfully.")

In [ ]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment = os.getenv(
    "AZURE_OPENAI_DEPLOYMENT",
    "gpt-5.4-ptu",
)

if not azure_endpoint or not azure_api_key:
    raise ValueError(
        "Missing Azure configuration. Check your .env file."
    )

set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

payment_model = OpenAIChatCompletionsModel(
    model=deployment,
    openai_client=azure_client,
)

print("Azure client and agent model configured.")

In [ ]:
import json

# Mock database simulating parking gate transactions and payment logs
TRANSACTION_DB = {
    "TXN-1001": {
        "gate": "Garage A North Gate", 
        "amount": 15.00, 
        "status": "Charged", 
        "refunded": False,
        "description": "Standard daily parking fee"
    },
    "TXN-1002": {
        "gate": "Garage B South Gate", 
        "amount": 25.00, 
        "status": "Double Charged", 
        "refunded": False,
        "description": "Duplicate gate terminal swipe error"
    },
    "TXN-1003": {
        "gate": "Surface Lot C Meter", 
        "amount": 8.50, 
        "status": "Charged", 
        "refunded": True,
        "description": "Overtime parking extension"
    }
}

def lookup_payment(transaction_id: str) -> str:
    """
    Finds the gate or meter transaction details from the database.
    
    Args:
        transaction_id (str): The unique transaction identifier (e.g., 'TXN-1001').
    """
    txn_id = transaction_id.strip().upper()
    txn = TRANSACTION_DB.get(txn_id)
    
    if not txn:
        return f"Error: Transaction ID '{transaction_id}' was not found in payment records."
    
    return json.dumps(txn)

def request_refund(transaction_id: str, amount: float, reason: str) -> str:
    """
    Submits a pending refund request. Enforces system guardrails: checks transaction existence,
    verifies amount does not exceed the charge, and prevents duplicate refunds[cite: 2].
    
    Args:
        transaction_id (str): The transaction ID to refund.
        amount (float): The monetary amount requested for refund.
        reason (str): The explanation for the refund request.
    """
    txn_id = transaction_id.strip().upper()
    txn_data = TRANSACTION_DB.get(txn_id)
    
    # Guardrail 1: Validate transaction exists
    if not txn_data:
        return f"Refund Failed: Transaction ID '{transaction_id}' is invalid or does not exist."
    
    # Guardrail 2: Prevent duplicate refunds
    if txn_data["refunded"]:
        return f"Refund Failed: Transaction '{transaction_id}' has already been refunded previously."
        
    # Guardrail 3: Verify amount does not exceed original charge
    if amount > txn_data["amount"]:
        return f"Refund Failed: Requested amount (${amount:.2f}) exceeds the original charge amount (${txn_data['amount']:.2f})."
        
    # Guardrail 4: High-limit check / approval routing (limits > $50 require human approval)
    if amount > 50.00:
        return f"PENDING HUMAN REVIEW: The refund request of ${amount:.2f} for transaction {txn_id} exceeds the automated threshold ($50.00) and has been routed to a human staff member for approval."

    # Process safe pending refund
    txn_data["refunded"] = True
    return f"SUCCESS: Pending refund of ${amount:.2f} for transaction {txn_id} successfully submitted and logged. Reason: {reason}"

In [ ]:
import json

# Mock database simulating parking gate transactions and payment logs
TRANSACTION_DB = {
    "TXN-1001": {
        "gate": "Garage A North Gate", 
        "amount": 15.00, 
        "status": "Charged", 
        "refunded": False,
        "description": "Standard daily parking fee"
    },
    "TXN-1002": {
        "gate": "Garage B South Gate", 
        "amount": 25.00, 
        "status": "Double Charged", 
        "refunded": False,
        "description": "Duplicate gate terminal swipe error"
    },
    "TXN-1003": {
        "gate": "Surface Lot C Meter", 
        "amount": 8.50, 
        "status": "Charged", 
        "refunded": True,
        "description": "Overtime parking extension"
    }
}

def lookup_payment(transaction_id: str) -> str:
    """
    Finds the gate or meter transaction details from the database.
    
    Args:
        transaction_id (str): The unique transaction identifier (e.g., 'TXN-1001').
    """
    txn_id = transaction_id.strip().upper()
    txn = TRANSACTION_DB.get(txn_id)
    
    if not txn:
        return f"Error: Transaction ID '{transaction_id}' was not found in payment records."
    
    return json.dumps(txn)

def request_refund(transaction_id: str, amount: float, reason: str) -> str:
    """
    Submits a pending refund request. Enforces system guardrails: checks transaction existence,
    verifies amount does not exceed the charge, and prevents duplicate refunds[cite: 2].
    
    Args:
        transaction_id (str): The transaction ID to refund.
        amount (float): The monetary amount requested for refund.
        reason (str): The explanation for the refund request.
    """
    txn_id = transaction_id.strip().upper()
    txn_data = TRANSACTION_DB.get(txn_id)
    
    # Guardrail 1: Validate transaction exists
    if not txn_data:
        return f"Refund Failed: Transaction ID '{transaction_id}' is invalid or does not exist."
    
    # Guardrail 2: Prevent duplicate refunds
    if txn_data["refunded"]:
        return f"Refund Failed: Transaction '{transaction_id}' has already been refunded previously."
        
    # Guardrail 3: Verify amount does not exceed original charge
    if amount > txn_data["amount"]:
        return f"Refund Failed: Requested amount (${amount:.2f}) exceeds the original charge amount (${txn_data['amount']:.2f})."
        
    # Guardrail 4: High-limit check / approval routing (limits > $50 require human approval)
    if amount > 50.00:
        return f"PENDING HUMAN REVIEW: The refund request of ${amount:.2f} for transaction {txn_id} exceeds the automated threshold ($50.00) and has been routed to a human staff member for approval."

    # Process safe pending refund
    txn_data["refunded"] = True
    return f"SUCCESS: Pending refund of ${amount:.2f} for transaction {txn_id} successfully submitted and logged. Reason: {reason}"

In [ ]:
lookup_payment_tool = function_tool(lookup_payment)
request_refund_tool = function_tool(request_refund)

print("Payment tools registered.")

In [ ]:
payment_agent = Agent(
    name="Payment Specialist",

    instructions="""
    You are a Parking Services payment specialist.

    Help customers with parking charges, duplicate payments,
    meter transaction failures, and refund requests.

    Rules:
    - Respond in English.
    - Use lookup_payment to check transaction records.
    - Never invent transaction details.
    - If the transaction ID is missing, check the conversation
      history before asking the customer for it.
    - If multiple transaction IDs were discussed and the intended
      transaction is unclear, ask for clarification.
    - Look up the transaction before requesting a refund.
    - Request a refund only when the customer explicitly asks.
    - Ask for the refund amount and reason when missing.
    - Explain tool errors accurately.
    - These tools use simulated data and do not transfer money.
    - Do not claim that a human employee was contacted.
    - Do not claim that a request was saved unless a tool
      actually recorded it.
    """,

    model=payment_model,

    tools=[
        lookup_payment_tool,
        request_refund_tool,
    ],
)

print("Payment Specialist created.")

In [ ]:
memory_db_path = Path("parking_sessions.db").resolve()
session_id = "payment-demo-001"

payment_session = SQLiteSession(
    session_id,
    str(memory_db_path),
)

print(f"Memory database: {memory_db_path}")
print(f"Session ID: {session_id}")

In [ ]:
import gradio as gr
from uuid import uuid4


def create_session_id():
    return f"payment-{uuid4().hex}"


async def gradio_chat(message, history, session_id):
    session = SQLiteSession(
        session_id,
        str(memory_db_path),
    )

    try:
        result = await Runner.run(
            payment_agent,
            input=message,
            session=session,
            max_turns=8,
        )

        return result.final_output
    finally:
        session.close()

In [ ]:
with gr.Blocks() as demo:
    session_state = gr.State(value=create_session_id)

    chatbot = gr.Chatbot(
        label="Parking Services",
    )

    gr.ChatInterface(
        fn=gradio_chat,
        chatbot=chatbot,
        additional_inputs=[session_state],
        title="Parking Services — Payment Support",
        description=(
            "Ask about parking transactions and simulated refund requests."
        ),
    )

    chatbot.clear(
        fn=create_session_id,
        inputs=[],
        outputs=[session_state],
    )

demo.launch(
    inbrowser=True,
    share=False,
    show_error=True,
)

In [ ]:
async def chat(message: str) -> str:
    result = await Runner.run(
        payment_agent,
        input=message,
        session=payment_session,
        max_turns=8,
    )

    return result.final_output

In [ ]:
response = await chat(
    "My name is Alex. My transaction ID is TXN-1002. "
    "Please check its amount and status."
)

print(response)

In [ ]:
response = await chat(
    "What transaction ID did I give you, and what was its amount?"
)

print(response)

In [ ]:
response = await chat(
    "What name did I give you?"
)

print(response)

In [ ]:
saved_items = await payment_session.get_items()

print(f"Saved history items: {len(saved_items)}")

for item in saved_items:
    print(item)

In [ ]:
response = await chat(
    "Before this restart, what name and transaction ID did I give you?"
)

print(response)

In [ ]:
payment_session = SQLiteSession(
    "payment-demo-002",
    str(memory_db_path),
)

response = await chat(
    "What transaction ID did I give you?"
)

print(response)

In [ ]:
await payment_session.clear_session()

print("Current session history cleared.")